# 🛠️ Data Cleaning Syntax Guide & Protocol

This notebook orchestrates the `DataCleaner` engine. The `CLEANING_CONFIGS` dictionary defines the deterministic pipeline for each dataset.

### 📜 The Configuration Ledger (Schema)

| Key | Type | Description | Available Options | Strict JSON Example |
|:---|:---|:---|:---|:---|
| `file` | `str` | Path to raw CSV. | Any valid local file path | `"Dataset/file.csv"` |
| `columns_to_drop` | `list` | Columns to remove immediately. | List of string column names | `["id_redundant", "useless_col"]` |
| `duplicate_strategy` | `str` | Row-level deduplication strategy. | `"keep_first"`, `"keep_last"`, `"keep_all"` | `"keep_first"` |
| `datetime_conversions` | `dict` | Datetime coercion rules. | Format: `"mixed"`, `"auto"`, or explicit string like `"%Y-%m-%d"` | `{"col": {"format": "auto"}}` |
| `text_cleaning` | `list` | Cols for NLP normalization (Unicode/Strip). | List of string column names | `["city", "description"]` |
| `value_mapping` | `dict` | Deterministic value replacements. | Mapping of `{"dirty_value": "clean_value"}` | `{"col": {"dirty_val": "clean_val"}}` |
| `foreign_keys` | `list` | Foreign Keys (Protected from statistical repair). | List of string column names | `["customer_id", "product_id"]` |
| `target_variables` | `list` | Labels (Protected from mutation/leakage). | List of string column names | `["churn_event"]` |
| `imputation_rules` | `dict` | Statistical missing value repair. | Strategy: `"simple"` (methods: `"mean"`, `"median"`, `"mode"`), `"advanced"` (methods: `"knn"`, `"mice"`), `"constant"` (requires `"fill_value"`). | `{"col": {"strategy": "simple", "method": "mode"}}` |
| `coercion_rules` | `dict` | Force-coerce mixed-type columns. | Target: `"numeric"`, `"datetime"`. Optional: `"strip_currency": True` | `{"col": {"target": "numeric", "strip_currency": True}}` |
| `cardinality_rules` | `dict` | Entropy-based rare-label collapsing. | Strategy: `"rare_label"`. Requires `"min_freq"` (float) | `{"col": {"strategy": "rare_label", "min_freq": 0.01}}` |
| `transforms` | `dict` | Mathematical skew-reduction. | Method: `"log1p"`, `"box_cox"`, `"yeo_johnson"` | `{"col": {"method": "log1p"}}` |
| `outlier_treatment` | `dict` | Outlier clipping (IQR/MAD/Winsor). | Method: `"cap_iqr"`, `"cap_mad"` (requires `"multiplier"`) or `"winsorize"` (requires `"limits"`) | `{"col": {"method": "winsorize", "limits": [0.05, 0.05]}}` |
| `leakage_rules` | `dict` | Temporal pruning of look-ahead features. | Keys: `"drop_cols"` (list), `"prediction_time_col"` (str) | `{"drop_cols": ["f1"], "prediction_time_col": "time_ts"}` |
| `association_rules` | `dict` | Pruning of redundant associated classes. | Keys: `"threshold"` (float), `"priority"` (list) | `{"threshold": 0.8, "priority": ["c1", "c2"]}` |
| `multicollinearity_threshold` | `float` | Spearman \|ρ\| drop threshold. | Float value between `0.0` and `1.0` | `0.90` |

### ⚙️ The Rigid Execution Pipeline
The engine enforces a strict sequential order to ensure state consistency:
1.  **Drop Columns**: Early removal of junk.
2.  **Deduplicate**: Row uniqueness enforcement.
3.  **Leakage Handling**: Removal of look-ahead features.
4.  **Datetime Conversions**: Type coercion for dates.
5.  **Coercion Cleansing**: Numeric forcing (stripping symbols).
6.  **Value Mapping**: Manual token cleanup.
7.  **Text Cleaning**: Unicode normalization & NLP prep.
8.  **Imputation**: Statistical repair (Simple/KNN/MICE).
9.  **Cardinality Grouping**: Collapsing rare labels.
10. **Transforms**: Skew reduction (Log/Box-Cox).
11. **Outlier Treatment**: Clipping extreme values.
12. **Resolve Categorical Association**: Pruning redundant categories.
13. **Resolve Multicollinearity**: Pruning redundant numeric features.
14. **Memory Optimization**: Final downcasting (Automatic).

### 🚫 Strict AI Rules for Automated Generation
1.  **Absolute Target Shield**: Never apply ANY statistical mutation (imputation_rules, transforms, outlier_treatment, cardinality_rules) to columns listed in `target_variables`.
2.  **ID Protection**: Never apply continuous transforms (`log1p`, etc.) to `foreign_keys` identifiers.
3.  **Sequential State**: Assume `columns_to_drop` and `duplicate_strategy` occur *before* any statistical treatment.
4.  **FK Imputation**: If imputing a `foreign_keys`, `constant` strategy with `valid_fk_placeholders` is mandatory.
5.  **Temporal State Awareness**: Outlier clipping (Step 11) occurs AFTER mathematical transforms (Step 10). If a `transform` is applied to a column, ensure the corresponding `outlier_treatment` multiplier accounts for the compressed, post-transform distribution.
6.  **No Hallucinations**: If a column's distribution is unknown, leave the treatment commented out with a `# [WARNING]`.
7. **Version & IP Protection**: Never assign `datetime_conversions`, `coercion_rules`, or `statistical continuous imputations` to semantic versioning columns or IP addresses. Always shield them by listing them under `foreign_keys` (to bypass outlier capping and skewness transforms) and use the 'constant' strategy under imputation_rules with safe string placeholders if missingness is encountered.


### 📋 Blank Configuration Template

Copy and paste this block into the `CLEANING_CONFIGS` dictionary below to add a new dataset to the pipeline.

```python
"dataset_name": {
    # ── Core Pipeline Settings ──────────────────────────────────────
    "file": "Dataset/your_file.csv",
    "columns_to_drop": [],
    "duplicate_strategy": "keep_first",

    # ── Key Protection & Target Shielding ───────────────────────────
    "foreign_keys": [],
    "target_variables": [],

    # ── Text, Value & Date Cleaning ─────────────────────────────────
    "datetime_conversions": {},
    "text_cleaning": [],
    "value_mapping": {},

    # ── Missing Value & Type Repair ─────────────────────────────────
    "imputation_rules": {},
    "coercion_rules": {},

    # ── Cardinality & Outlier Management ────────────────────────────
    "cardinality_rules": {},
    "transforms": {},
    "outlier_treatment": {},

    # ── Feature Pruning & Selection ──────────────────────────────────
    "leakage_rules": {},
    "association_rules": {},
    "multicollinearity_threshold": None
}
```

In [1]:
# ══════════════════════════════════════════════════════════════════════
#  Cell 1: Configuration Zone (LLM & Human Editable)
# ══════════════════════════════════════════════════════════════════════
# NOTE FOR ENGINEERS: Review all AI-generated #[WARNING] tags below. 
# These represent statistical trade-offs. To finalize a treatment, 
# uncomment the chosen line and remove the warning.


CLEANING_CONFIGS = {
    "olist_closed_deals_dataset": {
        "file": "Dataset/olist_closed_deals_dataset.csv",
        # [CRITICAL] High missingness (>90%) detected in these features.
        "columns_to_drop": ["has_company", "has_gtin", "average_stock", "declared_product_catalog_size"],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {
            # [WARNING] Convert string to datetime64 for time-series operations.
            "won_date": "auto"
        },
        "text_cleaning": ["business_segment", "lead_type", "lead_behaviour_profile", "business_type", "sdr_id", "sr_id"],
        "value_mapping": {},
        "foreign_keys": ["mql_id", "seller_id", "sdr_id", "sr_id"],
        "target_variables": [],
        "imputation_rules": {
            # [WARNING] Impute with Mode or add 'Unknown' category.
            "business_segment": {"strategy": "simple", "method": "mode"},
            # [WARNING] Impute with Mode or add 'Unknown' category.
            "lead_type": {"strategy": "simple", "method": "mode"},
            # [WARNING] Impute with Mode or add 'Unknown' category.
            "lead_behaviour_profile": {"strategy": "simple", "method": "mode"},
            # [WARNING] Impute with Mode or add 'Unknown' category.
            "business_type": {"strategy": "simple", "method": "mode"}
        },
        "transforms": {
            # [WARNING] Continuous non-negative revenue feature. Highly skewed (28.04).
            "declared_monthly_revenue": {"method": "log1p"}
        },
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    },
    "olist_customers_dataset": {
        "file": "Dataset/olist_customers_dataset.csv",
        # [CRITICAL] Unique ID — set as index or drop before ML modeling.
        "columns_to_drop": ["customer_unique_id"],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {},
        "text_cleaning": ["customer_city", "customer_state"],
        "value_mapping": {},
        "foreign_keys": ["customer_id", "customer_unique_id", "customer_zip_code_prefix"],
        "target_variables": [],
        "imputation_rules": {},
        "cardinality_rules": {
            # [WARNING] High-cardinality categorical column (4,119 unique values).
            "customer_city": {"strategy": "rare_label", "min_freq": 0.01}
        },
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    },
    "olist_geolocation_dataset": {
        "file": "Dataset/olist_geolocation_dataset.csv",
        "columns_to_drop": [],
        # [CRITICAL] Drop 261,831 fully duplicated rows to prevent spatial bias.
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {},
        "text_cleaning": [
            # [WARNING] Inconsistent casing/whitespace detected in city names.
            "geolocation_city", 
            "geolocation_state"
        ],
        "value_mapping": {},
        "foreign_keys": ["geolocation_zip_code_prefix"],
        "target_variables": [],
        "imputation_rules": {},
        "cardinality_rules": {
            # [WARNING] High-cardinality categorical column (8,011 unique values).
            "geolocation_city": {"strategy": "rare_label", "min_freq": 0.01}
        },
        # [WARNING] Latitude/Longitude are coordinate data. Avoid statistical capping.
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    },
    "olist_marketing_qualified_leads_dataset": {
        "file": "Dataset/olist_marketing_qualified_leads_dataset.csv",
        "columns_to_drop": [],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {
            # [WARNING] Convert string to datetime64 for time-series operations.
            "first_contact_date": "auto"
        },
        "text_cleaning": ["origin"],
        "value_mapping": {
            # [WARNING] Standardize 'unknown' tokens to true nulls.
            "origin": {"unknown": None}
        },
        "foreign_keys": ["mql_id", "landing_page_id"],
        "target_variables": [],
        "imputation_rules": {
            # [WARNING] Impute with Mode or add 'Unknown' category.
            "origin": {"strategy": "simple", "method": "mode"}
        },
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    },
    "olist_order_items_dataset": {
        "file": "Dataset/olist_order_items_dataset.csv",
        "columns_to_drop": [],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {
            # [WARNING] Convert string to datetime64 for time-series operations.
            "shipping_limit_date": "auto"
        },
        "text_cleaning": ["seller_id"],
        "value_mapping": {},
        "foreign_keys": ["order_id", "order_item_id", "product_id", "seller_id"],
        "target_variables": [],
        "imputation_rules": {},
        "transforms": {
            # [WARNING] Highly skewed (7.92).
            "price": {"method": "box_cox"},
            # [WARNING] Highly skewed (5.64).
            "freight_value": {"method": "log1p"}
        },
        "outlier_treatment": {
            # [WARNING] Outliers reviewed after transforms (Step 11 vs 10).
            "price": {"method": "cap_iqr", "multiplier": 1.5},
            "freight_value": {"method": "cap_iqr", "multiplier": 1.5}
        },
        "association_rules": {},
        # [INFO] Moderate threshold to resolve price/freight multicollinearity.
        "multicollinearity_threshold": 0.90
    },
    "olist_order_payments_dataset": {
        "file": "Dataset/olist_order_payments_dataset.csv",
        "columns_to_drop": [],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {},
        "text_cleaning": ["payment_type"],
        "value_mapping": {},
        "foreign_keys": ["order_id"],
        "target_variables": [],
        "imputation_rules": {},
        "transforms": {
            # [WARNING] Highly skewed (9.25).
            "payment_value": {"method": "log1p"}
        },
        "outlier_treatment": {
            "payment_value": {"method": "cap_iqr", "multiplier": 1.5}
        },
        "multicollinearity_threshold": None
    },
    "olist_order_reviews_dataset": {
        "file": "Dataset/olist_order_reviews_dataset.csv",
        # [CRITICAL] Unique ID and High Missingness (88.3%) in titles.
        "columns_to_drop": ["review_id", "review_comment_title"],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {
            # [WARNING] Convert string to datetime64 for time-series operations.
            "review_creation_date": "auto",
            # [WARNING] Convert string to datetime64 for time-series operations.
            "review_answer_timestamp": "auto"
        },
        "text_cleaning": ["review_comment_message"],
        "value_mapping": {
            # [WARNING] Standardize dirty missing tokens.
            "review_comment_title": {"": None, "-": None, "?": None},
            "review_comment_message": {"": None, "-": None, "?": None, "n/a": None}
        },
        "foreign_keys": ["order_id"],
        "target_variables": [],
        "imputation_rules": {},
        "cardinality_rules": {
            # [WARNING] High-cardinality categorical column (36,159 unique values).
            "review_comment_message": {"strategy": "rare_label", "min_freq": 0.01}
        },
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    },
    "olist_orders_dataset": {
        "file": "Dataset/olist_orders_dataset.csv",
        "columns_to_drop": [],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {
            "order_purchase_timestamp": "auto",
            "order_approved_at": "auto",
            "order_delivered_carrier_date": "auto",
            "order_delivered_customer_date": "auto",
            "order_estimated_delivery_date": "auto"
        },
        "text_cleaning": ["order_status"],
        "value_mapping": {},
        "foreign_keys": ["order_id", "customer_id"],
        "target_variables": [],
        "imputation_rules": {},
        "outlier_treatment": {},
        "leakage_rules": {
            # [WARNING] Temporal leakage candidates. Drop columns unavailable at purchase time.
            #"prediction_time_col": "order_purchase_timestamp",
            #"drop_cols": ["order_status", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date", "order_estimated_delivery_date"]
        },
        "multicollinearity_threshold": None
    },
    "olist_products_dataset": {
        "file": "Dataset/olist_products_dataset.csv",
        "columns_to_drop": [],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {},
        "text_cleaning": ["product_category_name"],
        "value_mapping": {
            # [WARNING] Review whether zero encodes missingness for weight.
            "product_weight_g": {0: None}
        },
        "foreign_keys": ["product_id"],
        "target_variables": [],
        "imputation_rules": {
            # [CRITICAL] Preserve FK nulls unless you have an explicit parent-table placeholder key.
            "product_category_name": {"strategy": "constant", "fill_value": "UNK_CATEGORY", "valid_fk_placeholders": ["UNK_CATEGORY"]},
            # [WARNING] Minor missing (1.9%). Impute with Median.
            "product_name_lenght": {"strategy": "simple", "method": "median"},
            # [WARNING] Minor missing (1.9%). Impute with Median.
            "product_description_lenght": {"strategy": "simple", "method": "median"},
            # [WARNING] Minor missing (1.9%). Impute with Median.
            "product_photos_qty": {"strategy": "simple", "method": "median"},
            # [WARNING] Minor missing (0.0%). Impute with Median.
            "product_weight_g": {"strategy": "simple", "method": "median"},
            # [WARNING] Minor missing (0.0%). Impute with Median.
            "product_length_cm": {"strategy": "simple", "method": "median"},
            # [WARNING] Minor missing (0.0%). Impute with Median.
            "product_height_cm": {"strategy": "simple", "method": "median"},
            # [WARNING] Minor missing (0.0%). Impute with Median.
            "product_width_cm": {"strategy": "simple", "method": "median"}
        },
        "transforms": {
            # [WARNING] Highly skewed (3.60).
            "product_weight_g": {"method": "log1p"},
            # [WARNING] Highly skewed (2.14).
            "product_height_cm": {"method": "box_cox"}
        },
        "outlier_treatment": {
            "product_description_lenght": {"method": "cap_iqr", "multiplier": 1.5},
            "product_weight_g": {"method": "cap_iqr", "multiplier": 1.5},
            "product_height_cm": {"method": "cap_iqr", "multiplier": 1.5}
        },
        "multicollinearity_threshold": None
    },
    "olist_sellers_dataset": {
        "file": "Dataset/olist_sellers_dataset.csv",
        "columns_to_drop": [],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {},
        "text_cleaning": ["seller_city", "seller_state"],
        "value_mapping": {},
        "foreign_keys": ["seller_id", "seller_zip_code_prefix"],
        "target_variables": [],
        "imputation_rules": {},
        "cardinality_rules": {
            # [WARNING] High-cardinality categorical column (611 unique values).
            "seller_city": {"strategy": "rare_label", "min_freq": 0.01}
        },
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    },
    "product_category_name_translation": {
        "file": "Dataset/product_category_name_translation.csv",
        # [CRITICAL] Unique ID — set as index or drop before ML modeling.
        "columns_to_drop": ["product_category_name_english"],
        "duplicate_strategy": "keep_first",
        "datetime_conversions": {},
        "text_cleaning": [],
        "value_mapping": {},
        "foreign_keys": ["product_category_name"],
        "target_variables": [],
        "imputation_rules": {},
        "outlier_treatment": {},
        "multicollinearity_threshold": None
    }
}

In [2]:
# ══════════════════════════════════════════════════════════════
#  Cell 2: Execution — Do not edit
# ══════════════════════════════════════════════════════════════

from data_cleaner_engine import CleaningOrchestrator

orchestrator = CleaningOrchestrator(CLEANING_CONFIGS)
orchestrator.execute_all()

Dataset,Original,Cleaned,Removed,Columns,Memory Change,Actions
olist_closed_deals_dataset,842,842,0 (0.0%),14 → 10 (-4),-443.2 KB,16
olist_customers_dataset,"50,000","50,000",0 (0.0%),5 → 5 (0),-5.4 MB,8
olist_geolocation_dataset,"50,000","28,946","-21,054 (42.1%)",5 → 5 (0),-5.7 MB,7
olist_marketing_qualified_leads_dataset,"8,000","8,000",0 (0.0%),4 → 4 (0),-849.5 KB,8
olist_order_items_dataset,"50,000","50,000",0 (0.0%),7 → 7 (0),-3.2 MB,11
olist_order_payments_dataset,"50,000","50,000",0 (0.0%),5 → 5 (0),-3.4 MB,7
olist_order_reviews_dataset,"50,000","50,000",0 (0.0%),7 → 5 (-2),-15.0 MB,9
olist_orders_dataset,"50,000","50,000",0 (0.0%),8 → 8 (0),-16.9 MB,10
olist_products_dataset,"32,951","32,951",0 (0.0%),9 → 9 (0),-1.8 MB,17
olist_sellers_dataset,"3,095","3,095",0 (0.0%),4 → 4 (0),-336.0 KB,7
